In [1]:
import sys
sys.path.append("..")
from pathlib import Path
from dataclasses import dataclass, fields
from concurrent.futures import ProcessPoolExecutor, Future

import pandas as pd

from rocnovo.metrcis.abnovobench import aa_match_metrics, aa_match_batch
from rocnovo.tokenizer.peptide import PTMPeptideTokenizer

DEFAULT_DATASETS = ["v1", "v2"]

@dataclass
class Metric:
    aa_precision: float
    aa_recall: float
    curve_auc: float
    pep_precision: float
    pep_recall: float
    ptm_precision: float
    ptm_recall: float
    full_accuracy: float

def calculate_metrics(df: pd.DataFrame, name: str):
    ptm_list = [
        "C+57.021",
        "M+15.995",
        "N+0.984",
        "Q+0.984"
    ]
    if name == "hc_pt":
        ptm_list.remove("C+57.021")
    
    tokenizer = PTMPeptideTokenizer(
        residues="massivekb",
        reverse=False
    )
    batch, n_pep, n_aa_true, n_aa_pred, n_ptm_true, n_ptm_pred = aa_match_batch(
        df["gt_peptide"].to_list(),
        df["pred_peptide"].to_list(),
        tokenizer.masses,
        ptm_list,
        0.5,
        0.1,
        "best"
    )
    metrics = aa_match_metrics(
        batch,
        n_pep,
        n_aa_true,
        n_aa_pred,
        n_ptm_true,
        n_ptm_pred,
        df["pred_score"]
    )
    return Metric(
        metrics["aa_precision"],
        metrics["aa_recall"],
        metrics["curve_auc"],
        metrics["pep_precision"],
        metrics["pep_recall"],
        metrics["ptm_precision"],
        metrics["ptm_recall"],
        (df["pred_peptide"] == df["gt_peptide"]).mean()
    )

def pipeline(
    root_dir: Path,
    target_datasets: list[str] = None,
    prefix: str = "",
    path_suffix: str=".csv"
) -> pd.DataFrame:
    raw_names = target_datasets if target_datasets is not None else DEFAULT_DATASETS
    folder_names = [f"{prefix}_{name}" if prefix else name for name in raw_names]
    
    dataset_item_sets: list[set[str]] = []
    valid_folders: list[str] = []
    
    for folder in folder_names:
        folder_path = root_dir / folder
        if folder_path.exists() and folder_path.is_dir():
            items = {p.stem for p in folder_path.iterdir() if p.suffix == path_suffix}
            dataset_item_sets.append(items)
            valid_folders.append(folder)
    
    if not dataset_item_sets:
        return pd.DataFrame()
    
    common_items = set.intersection(*dataset_item_sets)
    if not common_items:
        return pd.DataFrame()
    
    species_results = {item: {"species": item} for item in common_items}
    future_to_meta: dict[Future, dict[str, str]] = {}
    with ProcessPoolExecutor(max_workers=30) as executor:
        for item in sorted(common_items):
            for folder in valid_folders:
                subset_result_path = root_dir / folder / f"{item}{path_suffix}"

                if not subset_result_path.exists():
                    continue
                
                if path_suffix == ".csv":
                    df = pd.read_csv(
                        subset_result_path,
                        sep=",",
                        keep_default_na=False
                    )
                
                future = executor.submit(calculate_metrics, df, item)
                future_to_meta[future] = {
                    "item": item,
                    "folder": folder
                }
        
        for future, meta in future_to_meta.items():
            metric = future.result()
            item = meta["item"]
            folder = meta["folder"]
            
            for f in fields(metric):
                species_results[item][f"{folder}_{f.name}"] = getattr(metric, f.name)
    
    records = [species_results[item] for item in sorted(common_items)]
    df = pd.DataFrame(records)
    if not df.empty:
        df.loc[len(df)] = ["mean", *df.mean(numeric_only=True).values]
    
    return df

In [3]:
from rocnovo.common.io import normalize_path

novobench_df = pipeline(
    normalize_path("/data2/xp/RocNovo-Lightning/outputs/final"),
    ["novobench_results_large"]
)
novobench_df

100%|██████████| 28572/28572 [00:01<00:00, 18141.57it/s]


,species,novobench_results_large_aa_precision,novobench_results_large_aa_recall,novobench_results_large_curve_auc,novobench_results_large_pep_precision,novobench_results_large_pep_recall,novobench_results_large_ptm_precision,novobench_results_large_ptm_recall,novobench_results_large_full_accuracy
0,hc_pt,0.670393,0.670120,0.471608,0.512909,0.512909,0.742892,0.779689,0.511821
1,nine_species,0.832253,0.832956,0.632845,0.664847,0.664847,0.834906,0.765034,0.654137
2,seven_species,0.568318,0.570527,0.309827,0.367587,0.367587,0.543753,0.513859,0.356877
3,mean,0.690321,0.691201,0.471426,0.515114,0.515114,0.707184,0.686194,0.507612


In [4]:
print(novobench_df)

         species  novobench_results_large_aa_precision  \
0          hc_pt                              0.670393   
1   nine_species                              0.832253   
2  seven_species                              0.568318   
3           mean                              0.690321   

   novobench_results_large_aa_recall  novobench_results_large_curve_auc  \
0                           0.670120                           0.471608   
1                           0.832956                           0.632845   
2                           0.570527                           0.309827   
3                           0.691201                           0.471426   

   novobench_results_large_pep_precision  novobench_results_large_pep_recall  \
0                               0.512909                            0.512909   
1                               0.664847                            0.664847   
2                               0.367587                            0.367587   
3            